In [1]:
import time
import nidaqmx
from nidaqmx.constants import AcquisitionType
from nidaqmx.errors import DaqError
import datetime as dt
import numpy as np

In [2]:
task = nidaqmx.Task()

try:
    with nidaqmx.Task() as task:
        task.ai_channels.add_ai_voltage_chan("cDAQ2Mod4/ai0") # R1
        task.ai_channels.add_ai_voltage_chan("cDAQ2Mod4/ai1") # R2
        task.ai_channels.add_ai_voltage_chan("cDAQ2Mod4/ai2") # R3
        task.timing.cfg_samp_clk_timing(rate=1, sample_mode=AcquisitionType.CONTINUOUS)
        data = np.array(task.read(number_of_samples_per_channel=1))
        print(data.T)
        print(type(data))
except DaqError as e:
    print(f"Reading Error: {e}")

task.close()


[[0.15113084 0.51216732 0.22220999]]
<class 'numpy.ndarray'>


C:\Users\IPMU\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\nidaqmx\task.py:457: ResourceWarning: Attempted to close NI-DAQmx task of name "_unnamedTask<1>" but task was already closed.
  warnings.warn(


In [3]:
save_path = r"C:/Users/IPMU\Desktop/ipmu_DAQ/NIDAQ/"

filename = save_path + dt.datetime.now().strftime("%Y%m%d%H%M%S") + "_NIDAQ_NI9215.csv"
print(filename)

columnname = ['Abstime', 'Reltime','R1','R2','R3',]

try:
    with open(filename, mode="a") as f:
        print(*columnname, sep=", ", file=f)
except:
    pass

C:/Users/IPMU\Desktop/ipmu_DAQ/NIDAQ/20260714091223_NIDAQ_NI9215.csv


In [4]:
StarTime = time.time()
SAMPLING_RATE = 1
PERIOD = 1    # 100 ms

next_time = time.perf_counter()

with nidaqmx.Task() as task:
    task.ai_channels.add_ai_voltage_chan("cDAQ2Mod4/ai0")
    task.ai_channels.add_ai_voltage_chan("cDAQ2Mod4/ai1")
    task.ai_channels.add_ai_voltage_chan("cDAQ2Mod4/ai2")

    task.timing.cfg_samp_clk_timing(
        rate=SAMPLING_RATE,
        sample_mode=AcquisitionType.CONTINUOUS
    )

    while True:
        CurrentAbsTime = dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]
        CurrentRerTime = time.time() - StarTime
        arr = [CurrentAbsTime, CurrentRerTime]

        try:
            data = task.read(number_of_samples_per_channel=1)
            data = np.array(data).reshape(-1).tolist()
        except DaqError as e:
            print(f"Reading Error: {e}")
            next_time += PERIOD
            sleep_time = next_time - time.perf_counter()
            if sleep_time > 0:
                time.sleep(sleep_time)
            continue

        arr = arr + data
        print(arr)

        try:
            with open(filename, mode="a", encoding="utf-8") as f:
                print(*arr, sep=",", file=f)
        except FileNotFoundError:
            filename = save_path + dt.datetime.now().strftime("%Y%m%d%H%M%S") + ".csv"
            with open(filename, mode="a", encoding="utf-8") as f:
                print(*columnname, sep=",", file=f)
                print(*arr, sep=",", file=f)

        next_time += PERIOD
        sleep_time = next_time - time.perf_counter()
        if sleep_time > 0:
            time.sleep(sleep_time)

['2026-07-14 09:12:24.012', 0.0019953250885009766, 0.15431913400000002, 0.509005761, 0.220936067]


C:\Users\IPMU\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\nidaqmx\task.py:98: ResourceWarning: Task of name "_unnamedTask<0>" was not explicitly closed before it was destructed. Resources on the task device may still be reserved.
  warnings.warn(


['2026-07-14 09:12:25.026', 1.0164132118225098, 0.154956792, 0.510902697, 0.222528467]
['2026-07-14 09:12:26.028', 2.017937421798706, 0.153681476, 0.510270385, 0.221254547]
['2026-07-14 09:12:27.011', 3.001051187515259, 0.154956792, 0.510270385, 0.22189150700000002]
['2026-07-14 09:12:28.017', 4.007446527481079, 0.154956792, 0.510902697, 0.222528467]
['2026-07-14 09:12:29.018', 5.008489608764648, 0.153681476, 0.510270385, 0.22189150700000002]
['2026-07-14 09:12:30.015', 6.005273818969727, 0.15463796300000002, 0.510270385, 0.222209987]
['2026-07-14 09:12:31.019', 7.008746385574341, 0.15336264700000002, 0.509954229, 0.221254547]
['2026-07-14 09:12:32.017', 8.007447242736816, 0.15431913400000002, 0.509954229, 0.222209987]
['2026-07-14 09:12:33.014', 9.003982305526733, 0.153681476, 0.509954229, 0.221573027]
['2026-07-14 09:12:34.016', 10.005743265151978, 0.15431913400000002, 0.510270385, 0.22189150700000002]
['2026-07-14 09:12:35.023', 11.01339077949524, 0.15463796300000002, 0.511218853, 0

KeyboardInterrupt: 